In [21]:
# ==========================================
# PREPARACIÓN A: CARGA Y DIAGNÓSTICO
# ==========================================
import pandas as pd

def miles(n):
    """Formatea un entero con punto como separador de miles (ej: 103904 -> 103.904)."""
    return f"{n:,}".replace(",", ".")

# Lectura local prioritaria (reproducible sin internet) con fallback a GitHub
try:
    df = pd.read_csv("train.csv", index_col=0)
    print("Dataset cargado desde archivo local 'train.csv'.")
except FileNotFoundError:
    url = "https://raw.githubusercontent.com/egle06/airline-satisfaction-analysis/refs/heads/main/train.csv"
    df = pd.read_csv(url, index_col=0)
    print("Dataset cargado desde URL de GitHub.")

# 1. Estructura y tipos
print(f"\nFilas: {miles(df.shape[0])} | Columnas: {df.shape[1]}\n")
df.info()

# 2. Estadísticas descriptivas
print("\n--- describe() ---")
display(df.describe().T)

# 3. Nulos por columna
print("\n--- Nulos por columna ---")
nulos = df.isnull().sum()
display(nulos[nulos > 0])

# 4. Duplicados: filas completas e id repetidos
print(f"\nFilas completamente duplicadas: {df.duplicated().sum()}")
print(f"Filas con id repetido: {df['id'].duplicated().sum()}")

# 5. Consistencia de las variables de texto
print("\n--- Categorías de las variables de texto ---")
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

Dataset cargado desde URL de GitHub.

Filas: 103.904 | Columnas: 24

<class 'pandas.core.frame.DataFrame'>
Index: 103904 entries, 0 to 103903
Data columns (total 24 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 103904 non-null  int64  
 1   Gender                             103904 non-null  object 
 2   Customer Type                      103904 non-null  object 
 3   Age                                103904 non-null  int64  
 4   Type of Travel                     103904 non-null  object 
 5   Class                              103904 non-null  object 
 6   Flight Distance                    103904 non-null  int64  
 7   Inflight wifi service              103904 non-null  int64  
 8   Departure/Arrival time convenient  103904 non-null  int64  
 9   Ease of Online booking             103904 non-null  int64  
 10  Gate location                      10390

,count,mean,std,min,25%,50%,75%,max
id,103904.0,64924.210502,37463.812252,1.0,32533.75,64856.5,97368.25,129880.0
Age,103904.0,39.379706,15.114964,7.0,27.00,40.0,51.00,85.0
Flight Distance,103904.0,1189.448375,997.147281,31.0,414.00,843.0,1743.00,4983.0
Inflight wifi service,103904.0,2.729683,1.327829,0.0,2.00,3.0,4.00,5.0
Departure/Arrival time convenient,103904.0,3.060296,1.525075,0.0,2.00,3.0,4.00,5.0
Ease of Online booking,103904.0,2.756901,1.398929,0.0,2.00,3.0,4.00,5.0
Gate location,103904.0,2.976883,1.277621,0.0,2.00,3.0,4.00,5.0
Food and drink,103904.0,3.202129,1.329533,0.0,2.00,3.0,4.00,5.0
Online boarding,103904.0,3.250375,1.349509,0.0,2.00,3.0,4.00,5.0
Seat comfort,103904.0,3.439396,1.319088,0.0,2.00,4.0,5.00,5.0



--- Nulos por columna ---


,0
Arrival Delay in Minutes,310



Filas completamente duplicadas: 0
Filas con id repetido: 0

--- Categorías de las variables de texto ---

Gender:
Gender
Female    52727
Male      51177
Name: count, dtype: int64

Customer Type:
Customer Type
Loyal Customer       84923
disloyal Customer    18981
Name: count, dtype: int64

Type of Travel:
Type of Travel
Business travel    71655
Personal Travel    32249
Name: count, dtype: int64

Class:
Class
Business    49665
Eco         46745
Eco Plus     7494
Name: count, dtype: int64

satisfaction:
satisfaction
neutral or dissatisfied    58879
satisfied                  45025
Name: count, dtype: int64


## Preparación estructural y división en Train / Test
Para evitar fuga de información (*data leakage*), antes de dividir el dataset solo se aplican limpiezas estructurales que no calculan estadísticos poblacionales (eliminación de duplicados, normalización de texto y codificación de variables). La imputación de valores nulos y el tratamiento de ceros en escalas se realizan **después** del `train_test_split`, ajustando la mediana y la moda únicamente sobre `X_train`.

In [22]:
# ==========================================
# PREPARACIÓN B: LIMPIEZA ESTRUCTURAL Y ARMADO DE X E y
# ==========================================
df_limpio = df.copy()
print(f"Filas iniciales: {miles(len(df_limpio))}")

# 1. Duplicados
df_limpio = df_limpio.drop_duplicates(subset='id').reset_index(drop=True)
print(f"Tras eliminar duplicados: {miles(len(df_limpio))} filas")

# 2. Normalización de texto (mayúsculas inconsistentes)
for col in ['Customer Type', 'Type of Travel']:
    df_limpio[col] = df_limpio[col].str.strip().str.title()

# 3. Definición de Target (y) y Características (X)
y = (df_limpio['satisfaction'] == 'satisfied').astype(int)
X = df_limpio.drop(columns=['id', 'satisfaction'], errors='ignore')

# 4. Variables categóricas a numéricas (0/1)
X = pd.get_dummies(X, columns=['Gender', 'Customer Type', 'Type of Travel', 'Class'],
                   drop_first=True, dtype=int)

print(f"\nX: {miles(X.shape[0])} filas x {X.shape[1]} columnas")
print(f"y: {miles(y.shape[0])} valores")
print(f"\nBalance del target:\n{y.value_counts(normalize=True).round(3)}")

Filas iniciales: 103.904
Tras eliminar duplicados: 103.904 filas

X: 103.904 filas x 23 columnas
y: 103.904 valores

Balance del target:
satisfaction
0    0.567
1    0.433
Name: proportion, dtype: float64


In [26]:
# ==========================================
# PASO 1: DIVISIÓN TRAIN / TEST E IMPUTACIÓN SIN DATA LEAKAGE
# ==========================================
from sklearn.model_selection import train_test_split

# 1. División estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train.copy()
X_test = X_test.copy()

print(f"Train: {miles(X_train.shape[0])} filas ({X_train.shape[0] / len(X) * 100:.0f} %)")
print(f"Test:  {miles(X_test.shape[0])} filas ({X_test.shape[0] / len(X) * 100:.0f} %)")

# 2. Imputación de nulos en 'Arrival Delay in Minutes' (mediana calculada SOLO en train)
mediana_llegada = X_train['Arrival Delay in Minutes'].median()
# Relleno los nulos con la mediana_llegada tanto en train como en test
X_train['Arrival Delay in Minutes'] = X_train['Arrival Delay in Minutes'].fillna(mediana_llegada).astype(int)
X_test['Arrival Delay in Minutes'] = X_test['Arrival Delay in Minutes'].fillna(mediana_llegada).astype(int)
print(f"\nMediana de 'Arrival Delay in Minutes' (calculada en Train): {mediana_llegada} min")
print(f"Nulos restantes -> Train: {X_train.isnull().sum().sum()} | Test: {X_test.isnull().sum().sum()}")

# 3. Tratamiento de ceros en escalas de servicio
cols_servicio = ['Inflight wifi service', 'Departure/Arrival time convenient',
                 'Ease of Online booking', 'Gate location', 'Food and drink',
                 'Online boarding', 'Seat comfort', 'Inflight entertainment',
                 'On-board service', 'Leg room service', 'Baggage handling',
                 'Checkin service', 'Inflight service', 'Cleanliness']

#calculo la cantidad de ceros en train y en test para los servicios
ceros_train = (X_train[cols_servicio] == 0).sum().sum()
ceros_test = (X_test[cols_servicio] == 0).sum().sum()
print(f"\nCeros a tratar -> Train: {miles(ceros_train)} | Test: {miles(ceros_test)}")

# Calculo la moda en SOLO en train sin contar ceros y luego reemplazo los ceros por el valor de moda_train tanto en train como en test
for col in cols_servicio:
    moda_train = X_train[col][X_train[col] != 0].mode()[0]
    X_train[col] = X_train[col].mask(X_train[col] == 0, moda_train)
    X_test[col] = X_test[col].mask(X_test[col] == 0, moda_train)

print(f"Ceros restantes -> Train: {(X_train[cols_servicio] == 0).sum().sum()} | Test: {(X_test[cols_servicio] == 0).sum().sum()}")

Train: 83.123 filas (80 %)
Test:  20.781 filas (20 %)

Mediana de 'Arrival Delay in Minutes' (calculada en Train): 0.0 min
Nulos restantes -> Train: 0 | Test: 0

Ceros a tratar -> Train: 12.720 | Test: 3.212
Ceros restantes -> Train: 0 | Test: 0


In [25]:
# ==========================================
# PASO 2: ENTRENAMIENTO Y EVALUACIÓN DEL MODELO BASE (ÁRBOL SIN LÍMITE)
# ==========================================
from sklearn.tree import DecisionTreeClassifier

modelo_base = DecisionTreeClassifier(random_state=42)
modelo_base.fit(X_train, y_train)

score_train = modelo_base.score(X_train, y_train)
score_test = modelo_base.score(X_test, y_test)
referencia = 1 - y_test.mean()

print(f"Profundidad del árbol: {modelo_base.get_depth()} | Cantidad de hojas: {miles(modelo_base.get_n_leaves())}\n")
print(f"Score en TRAIN: {score_train:.4f} ({score_train * 100:.2f} %)")
print(f"Score en TEST:  {score_test:.4f} ({score_test * 100:.2f} %)")
print(f"Diferencia (train - test): {(score_train - score_test) * 100:.2f} puntos porcentuales")
print(f"Referencia (predecir siempre 'no satisfecho'): {referencia:.4f} ({referencia * 100:.2f} %)")

Profundidad del árbol: 36 | Cantidad de hojas: 3.784

Score en TRAIN: 1.0000 (100.00 %)
Score en TEST:  0.9362 (93.62 %)
Diferencia (train - test): 6.38 puntos porcentuales
Referencia (predecir siempre 'no satisfecho'): 0.5667 (56.67 %)


### Selección de hiperparámetros con Validación Cruzada (`cross_val_score`)
Para elegir la profundidad óptima (`max_depth`) sin contaminar ni introducir sesgo de optimización sobre `X_test`, se evalúa cada candidato mediante **Validación Cruzada Estratificada de 5 particiones (5-Fold CV)** exclusivamente dentro de `X_train`. El conjunto de test se reserva para evaluar una única vez el árbol ganador.

In [27]:
# ==========================================
# PASO 3: BÚSQUEDA DE max_depth CON VALIDACIÓN CRUZADA (SOLO EN TRAIN)
# ==========================================
from sklearn.model_selection import cross_val_score

resultados_cv = []
for prof in [3, 5, 8, 10, 15, 20, None]:
    m = DecisionTreeClassifier(max_depth=prof, random_state=42)

    # Validación cruzada de 5 folds solo con datos de entrenamiento
    scores_cv = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)

    # Ajuste en train para comparar cuánto memoriza en entrenamiento vs. validación CV
    m.fit(X_train, y_train)
    train_acc = m.score(X_train, y_train)
    cv_mean = scores_cv.mean()

    resultados_cv.append({
        'max_depth': 'sin límite' if prof is None else prof,
        'Train': round(train_acc, 4),
        'CV Validación (Media)': round(cv_mean, 4),
        'CV Desvío (Std)': round(scores_cv.std(), 4),
        'Brecha (Train - CV)': round(train_acc - cv_mean, 4)
    })

df_cv = pd.DataFrame(resultados_cv)
display(df_cv)

# Elegimos el mejor max_depth según 'CV Validación (Media)' (15) y evaluamos recién ahora en Test
mejor_profundidad = 15
arbol_podado = DecisionTreeClassifier(max_depth=mejor_profundidad, random_state=42)
arbol_podado.fit(X_train, y_train)

podado_train = arbol_podado.score(X_train, y_train)
podado_test = arbol_podado.score(X_test, y_test)

print(f"\n--- Evaluación final en Test del Árbol Podado (max_depth={mejor_profundidad}) ---")
print(f"Train: {podado_train:.4f} ({podado_train*100:.2f} %) | Test: {podado_test:.4f} ({podado_test*100:.2f} %)")
print(f"Brecha (Train - Test): {(podado_train - podado_test)*100:.2f} p.p.")

,max_depth,Train,CV Validación (Media),CV Desvío (Std),Brecha (Train - CV)
0,3,0.8740,0.8740,0.0015,-0.0000
1,5,0.8965,0.8958,0.0018,0.0007
2,8,0.9298,0.9268,0.0020,0.0031
3,10,0.9434,0.9355,0.0012,0.0079
4,15,0.9725,0.9425,0.0016,0.0299
5,20,0.9890,0.9381,0.0016,0.0509
6,sin límite,1.0000,0.9345,0.0017,0.0655



--- Evaluación final en Test del Árbol Podado (max_depth=15) ---
Train: 0.9725 (97.25 %) | Test: 0.9462 (94.62 %)
Brecha (Train - Test): 2.62 p.p.


In [28]:
# ==========================================
# PASO 4: COMPARACIÓN CON RANDOM FOREST
# ==========================================
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_cv = cross_val_score(rf, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1).mean()
rf_train = rf.score(X_train, y_train)
rf_test = rf.score(X_test, y_test)

# Tabla resumen comparativa de todos los modelos
comparativa = pd.DataFrame([
    {
        'Modelo': '1. Árbol sin límite (Base)',
        'Train': round(score_train, 4),
        'CV (5-Fold)': df_cv.loc[df_cv['max_depth'] == 'sin límite', 'CV Validación (Media)'].values[0],
        'Test': round(score_test, 4),
        'Brecha (Train - Test)': round(score_train - score_test, 4)
    },
    {
        'Modelo': '2. Árbol podado (max_depth=15)',
        'Train': round(podado_train, 4),
        'CV (5-Fold)': df_cv.loc[df_cv['max_depth'] == 15, 'CV Validación (Media)'].values[0],
        'Test': round(podado_test, 4),
        'Brecha (Train - Test)': round(podado_train - podado_test, 4)
    },
    {
        'Modelo': '3. Random Forest (100 árboles, max_depth=15)',
        'Train': round(rf_train, 4),
        'CV (5-Fold)': round(rf_cv, 4),
        'Test': round(rf_test, 4),
        'Brecha (Train - Test)': round(rf_train - rf_test, 4)
    }
])

display(comparativa)

,Modelo,Train,CV (5-Fold),Test,Brecha (Train - Test)
0,1. Árbol sin límite (Base),1.0000,0.9345,0.9362,0.0638
1,2. Árbol podado (max_depth=15),0.9725,0.9425,0.9462,0.0262
2,"3. Random Forest (100 árboles, max_depth=15)",0.9769,0.9508,0.9524,0.0246


## Análisis crítico y conclusiones

### 1. Resumen comparativo de modelos

| Modelo | Train | CV (5-Fold) | Test | Brecha (Train - Test) |
|---|:---:|:---:|:---:|:---:|
| **1. Árbol sin límite (Base)** | 100,00 % (1,0000) | 93,45 % (0,9345) | 93,62 % (0,9362) | 6,38 p.p. (0,0638) |
| **2. Árbol podado (`max_depth = 15`)** | 97,25 % (0,9725) | 94,25 % (0,9425) | 94,62 % (0,9462) | 2,62 p.p. (0,0262) |
| **3. Random Forest (100 árboles, `max_depth = 15`)** | **97,69 % (0,9769)** | **95,08 % (0,9508)** | **95,24 % (0,9524)** | **2,46 p.p. (0,0246)** |
| **Referencia (predecir siempre «no satisfecho»)** | — | — | 56,67 % (0,5667) | — |

---

### 2. Diagnóstico del modelo base (Árbol sin límite)
* **¿Hay overfitting? Sí, de forma moderada.** Sin restricción de profundidad, el árbol crece hasta memorizar el 100,00 % del conjunto de entrenamiento, mientras que en validación cruzada (93,45 %) y en prueba (93,62 %) cae 6,38 puntos porcentuales. Aun así, supera ampliamente la referencia base de predecir siempre la clase mayoritaria (56,67 %), confirmando que aprendió patrones reales que se repiten en pasajeros nuevos.
* **¿Hay underfitting? No.** Los puntajes tanto en entrenamiento como en validación y prueba son elevados; el modelo tiene capacidad de sobra para capturar la estructura de los datos.

---

### 3. Prevención de fuga de datos y sesgo de optimización
* **Preprocesamiento sin *data leakage*:** Tanto la mediana de `Arrival Delay in Minutes` como la moda de las 14 escalas de servicio fueron calculadas exclusivamente sobre `X_train` luego del `train_test_split` estratificado, aplicando esos parámetros fijos sobre `X_test`.
* **Selección mediante Validación Cruzada (`cross_val_score`):** En lugar de exponer el conjunto de prueba para elegir `max_depth`, se utilizó validación cruzada de 5 particiones sobre `X_train`. Allí se identificó `max_depth = 15` como el punto óptimo de generalización (94,25 % en CV) antes de que el rendimiento de validación caiga por memorización.

---

### 4. Efecto de la poda y salto a Random Forest
1. **Árbol podado (`max_depth = 15`):** Limitar la profundidad redujo la brecha entre entrenamiento y prueba a menos de la mitad (de 6,38 p.p. a 2,62 p.p.) y elevó la exactitud en el conjunto de prueba de 93,62 % a 94,62 % (+1,00 p.p.).
2. **Random Forest (`n_estimators = 100`, `max_depth = 15`):** Al promediar 100 árboles entrenados sobre submuestras aleatorias, se redujo aún más la varianza. Alcanzó el mejor desempeño tanto en validación cruzada (95,08 %) como en test (95,24 %), logrando además la menor brecha de sobreajuste de todo el experimento (2,46 p.p.).
3. **Próximos pasos:** Complementar la métrica de *accuracy* con **Matriz de confusión, Precisión, Recall y F1-Score** para evaluar el impacto de falsos positivos y falsos negativos en la detección de pasajeros insatisfechos.